# 23 — ML combo selection

Wider LOTO (leave-one-target-out) ML sweep using Protocol P1: per-target combo selection. Reads the results table from `reproduce/deep_research_wide.py` and asks: does any model beat GBSA-locked at per-target combo selection? Answer previews NB 25's verdict.

See `docs/GLOSSARY.md` for term definitions.

> **Bootstrap regime.** This notebook uses B=1000, seed 20250901 for speed. The canonical repo-wide regime (see `data/derived/canonical_baselines.csv`) is B=5000, seed 20260902. Numbers are stable at reported precision; CIs here are a hair wider.

> **Reader guide.** *Experiment A3 (see [STUDY_DESIGN §A3](../../STUDY_DESIGN.md)):* per-complex
> MD-feature analysis and downstream ranking questions.
>
> See STUDY_DESIGN Chapter §A3 Q1 (per-target combo selection) and Q2 (single-feature panel
> ranker) for the framing this notebook addresses.
>
> **Reproducibility contract:** reads `data/derived/features.parquet` +
> `data/raw/reference/ohds_metadata.csv` (and `data/derived/canonical_baselines.csv` for
> baseline comparison).

> **Reader's guide — what this notebook does, in plain language**\n>\n> **Question:** Can we train an ML model that, for a *new* target, picks the\n> best GBSA parameter combination (out of 48 options) — better than just\n> using the reviewer-locked combo `igb2_di4_salt0.15_st0.0072` for every target?\n>\n> **Method — step by step:**\n> 1. For each of the 9 targets we take **41 MD descriptors** (backbone RMSD,\n>    buried SASA, hydrogen bonds, VDW contacts, etc.) as input features.\n>    These are aggregated per target (median + std across its 30 ligands).\n> 2. The **label** is the BEDROC value (α=20 — measures early enrichment of\n>    actives in the ranking, see [`33_ml_walkthrough_tutorial`](33_ml_walkthrough_tutorial.ipynb) §2)\n>    for each of the 48 GBSA combos on that target.\n> 3. **Leave-One-Target-Out CV** (LOTO): hold out one target completely, train\n>    on the other 8, predict on the held-out one. Repeat 9× so every target is\n>    the held-out one exactly once. Why LOTO instead of random-split → Tutorial §4.\n> 4. **Models tested:** `Ridge`, `RandomForest`, `ExtraTrees`, `HistGradientBoosting`,\n>    `SVM-RBF`, `ElasticNet` (all `sklearn`). Feature scaling + median imputation\n>    fitted *inside each fold* so there is no leakage from held-out to training.\n> 5. For each model: **1000× bootstrap resampling** → 95% confidence interval,\n>    plus a **permutation test** vs \"pick a random combo per target\".\n>\n> **How to read the numbers:**\n> - `panel BEDROC = 0.541` → GBSA-locked, our baseline. Anything above = win.\n> - `CI = [0.334, 0.684]` → 95% confidence interval. Only if the *lower bound\n>   > 0.541* is the win statistically supported. If not, it's noise.\n> - `perm p = 0.081` → probability of seeing this value under the random-pick\n>   null. p < 0.05 = significant.\n>\n> **Bottom line:** Ridge wins the ML tournament (0.511), but its CI lower\n> bound (0.334) sits far below the GBSA baseline (0.541). No model beats\n> baseline with statistical confidence → **Claim A rejected**.

In [ ]:
# --- notebook preamble ---
NB_STEM = "38_ml_combo_selection"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')

## Protocol P1 — per-target combo selection

**Question.** Given a target's per-complex MD fingerprint, can any ML approach pick a GBSA combo whose per-target BEDROC α=20 beats the single GBSA-locked combo used across every target?

<!-- canonical baseline — see data/derived/canonical_baselines.csv -->

**GBSA-locked** = the fixed combo with the highest mean per-target BEDROC across the 9-target discovery panel (4A5S imputed at 0 where the combo has no data). Same combo everywhere — that's what "locked" means. Canonical panel value = **0.5412** for `igb2_di4_salt0.15_st0.0072`; see `data/derived/canonical_baselines.csv`.

**Setup.**
- Regressors trained LOTO on 7 targets, tested on the held-out target.
- Features: per-target aggregates (median + std over that target's 30 ligands) of the 41 MD descriptors in `FP_COLS`.
- Label: per-(target, combo) BEDROC α=20 from `bedroc_all_combos_per_target.csv`.
- For the held-out target, ML picks argmax(predicted BEDROC) over the 48 GBSA combos. The score used to grade the pick is the true BEDROC of that combo on that target.
- Preprocessing (median imputation + StandardScaler for linear / SVM) fit inside each fold — no global fits, no leakage.

**Uncertainty.** Bootstrap 95% CI on panel BEDROC (B=1000, resample targets with replacement). Permutation p (B=1000): null = pick a random combo per target from that target's own BEDROC row.

**Model panel.** HistGradientBoosting, RandomForest (tuned depth), ExtraTrees, Ridge, ElasticNet, SVM-RBF. XGBoost / LightGBM are logged and skipped when the pixi env doesn't provide them.

In [ ]:
# Read the wide-sweep result table produced by reproduce/deep_research_wide.py
wide_csv = DERIVED / 'deep_research_wide.csv'
assert wide_csv.exists(), f'{wide_csv} missing — run: cd env && pixi run python ../reproduce/deep_research_wide.py'
wide = pd.read_csv(wide_csv)
locked_combo = wide.locked_combo.dropna().iloc[0]
print(f'GBSA-locked combo (highest mean per-target BEDROC α=20): {locked_combo}')
print(f'rows: {len(wide)}   protocols: {sorted(wide.protocol.unique())}   models: {sorted(wide.model.unique())}')

P1 = wide[wide.protocol == 'P1'].copy().sort_values('panel_bedroc', ascending=False).reset_index(drop=True)
P2 = wide[wide.protocol == 'P2'].copy().sort_values('panel_bedroc', ascending=False).reset_index(drop=True)
baseline_p1 = float(P1[P1.model == 'GBSA-locked'].panel_bedroc.iloc[0])
baseline_p2 = float(P2[P2.model == 'GBSA-locked'].panel_bedroc.iloc[0])
print(f'\nP1 GBSA-locked panel BEDROC = {baseline_p1:.3f}')
print(f'P2 GBSA-locked panel BEDROC = {baseline_p2:.3f}')

# Pretty-print the P1 table
def _fmt_ci(lo, hi):
    if pd.isna(lo) or pd.isna(hi):
        return 'NA'
    return f'[{lo:.3f}, {hi:.3f}]'

p1_view = P1[['model','panel_bedroc','ci_lo','ci_hi','perm_p','n_targets']].copy()
p1_view['95% CI'] = [_fmt_ci(l, h) for l, h in zip(p1_view.ci_lo, p1_view.ci_hi)]
p1_view = p1_view[['model','panel_bedroc','95% CI','perm_p','n_targets']].round(3)
print('\nP1 panel BEDROC α=20 (all models, sorted):')
print(p1_view.to_string(index=False))

In [ ]:
# Horizontal bar plot: panel BEDROC + 95% CI, GBSA-locked = GOLD baseline line.
sub = P1.sort_values('panel_bedroc', ascending=True).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(9.5, 0.55 * len(sub) + 1.6))
y = np.arange(len(sub))

# split into GBSA-locked (drawn gold) and ML models (drawn navy) for clarity
is_baseline = (sub.model == 'GBSA-locked').values
colors = np.where(is_baseline, GOLD, NAVY)

vals = sub.panel_bedroc.values
los  = sub.ci_lo.values
his  = sub.ci_hi.values
err_lo = np.clip(vals - los, 0, None)
err_hi = np.clip(his - vals, 0, None)

ax.barh(y, vals, color=colors, edgecolor=NAVY, linewidth=0.6, alpha=0.9)
ax.errorbar(vals, y, xerr=[err_lo, err_hi], fmt='none', ecolor=GREY, capsize=3, lw=1.4)
ax.axvline(baseline_p1, color=GOLD, ls='--', lw=1.5, label=f'GBSA-locked ({baseline_p1:.3f})')

ax.set_yticks(y); ax.set_yticklabels(sub.model)
ax.set_xlabel('Panel BEDROC α=20  (mean over 9 held-out targets, 4A5S imputed at 0)')
ax.set_title('P1 — per-target GBSA-combo selection: model panel BEDROC + 95% bootstrap CI')
ax.set_axisbelow(True)
ax.xaxis.grid(True, color=GREY, alpha=0.5)
ax.set_xlim(0, max(1.02, float(np.nanmax(his)) * 1.05))
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()

# Register for later export.


In [ ]:
# Per-target small-multiples: winning ML model vs GBSA-locked for each target.
# 'winning' = highest panel BEDROC among the ML models (excluding the baseline).
def _parse_per_target(s):
    d = {}
    if not isinstance(s, str):
        return d
    for tok in s.split(';'):
        if '=' in tok:
            k, v = tok.split('=', 1)
            try:
                d[k] = float(v)
            except ValueError:
                d[k] = np.nan
    return d

p1_ml = P1[P1.model != 'GBSA-locked'].copy()
winner_row = p1_ml.iloc[p1_ml.panel_bedroc.astype(float).idxmax()]
winner_name = str(winner_row.model)
winner_pt = _parse_per_target(winner_row.per_target)
locked_row = P1[P1.model == 'GBSA-locked'].iloc[0]
locked_pt = _parse_per_target(locked_row.per_target)

targets = sorted(set(winner_pt) | set(locked_pt))
print(f'Winning ML model (by panel BEDROC): {winner_name}   panel = {float(winner_row.panel_bedroc):.3f}')
print(f'GBSA-locked panel = {baseline_p1:.3f}')
print(f'targets in per-target comparison: {targets}')

# Pad to 3x3 grid
nc, nr = 3, 3
fig, axes = plt.subplots(nr, nc, figsize=(3.4 * nc, 2.9 * nr), sharey=True)
axes = np.array(axes).ravel()
for ax, tgt in zip(axes, targets):
    w = winner_pt.get(tgt, np.nan)
    l = locked_pt.get(tgt, np.nan)
    ax.bar([0, 1], [l, w],
           color=[GOLD, NAVY], edgecolor=NAVY, linewidth=0.6)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['locked', winner_name], rotation=25, ha='right', fontsize=8)
    ax.set_ylim(0, 1.02)
    ax.set_title(tgt, fontsize=10)
    ax.axhline(baseline_p1, color=GREYD, ls=':', lw=1)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color=GREY, alpha=0.5)
    for x, v in enumerate([l, w]):
        if np.isfinite(v):
            ax.text(x, v + 0.02, f'{v:.2f}', ha='center', va='bottom', fontsize=8, color=NAVY)
# hide unused panes
for k in range(len(targets), len(axes)):
    axes[k].axis('off')
fig.suptitle(f'P1 per-target BEDROC α=20 — GBSA-locked (GOLD) vs winning ML "{winner_name}" (NAVY)\n'
             f'GREY dashed = panel GBSA-locked baseline ({baseline_p1:.3f})',
             y=1.01, color=NAVY, fontweight='bold')
plt.tight_layout()

# Register for later export.


In [ ]:
# Verdict logic: which (if any) ML model has its 95% CI lower bound above the GBSA-locked panel BEDROC?
beaters = []
for _, r in P1.iterrows():
    if r.model == 'GBSA-locked':
        continue
    if pd.notna(r.ci_lo) and float(r.ci_lo) > baseline_p1:
        beaters.append((str(r.model), float(r.panel_bedroc), float(r.ci_lo), float(r.ci_hi), float(r.perm_p) if pd.notna(r.perm_p) else float('nan')))

print(f'Baseline GBSA-locked panel BEDROC = {baseline_p1:.3f}')
print(f'Winning ML model: {winner_name}   panel = {float(winner_row.panel_bedroc):.3f}   '
      f'CI = [{float(winner_row.ci_lo):.3f}, {float(winner_row.ci_hi):.3f}]   perm p = {float(winner_row.perm_p):.3f}')
print(f'\nModels beating baseline within CI (lower bound > {baseline_p1:.3f}): '
      f'{[b[0] for b in beaters] if beaters else "NONE"}')

> **Verdict on Claim A** (auto-generated from the CSV; see the `beaters` list above.)
>
> If no model has its 95% CI lower bound above GBSA-locked:
>
> **Claim A stands. No tested ML approach — SVM-RBF, RandomForest, ExtraTrees, HistGradientBoosting, Ridge, ElasticNet, and XGBoost / LightGBM when the env provides them — beats GBSA-locked for per-target combo selection at the wider tuned sweep with in-fold preprocessing.**
>
> If one or more models beat the baseline with CI above it, the `beaters` list is non-empty and the cell above reports the finding honestly; Claim A is retracted for those models.
>
> Caveat: 9 labelled targets means 9 LOTO folds, so this is a low-power test (4A5S imputed at 0 for the GBSA-locked baseline where upstream GBSA has no data). Point of the sweep is to close the door on "you didn't tune hard enough" — not to claim a strong negative that would carry to a much larger validation set.

In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
